# Notebook 1 — Bigram Language Model on `names.txt`

이 노트북의 목표는 **가장 기본적인 문자 단위 language model**을 만드는 것입니다.

- 입력: 현재 문자 1개
- 출력: 다음 문자 1개
- 즉, **bigram** 확률표를 학습합니다.

전체 구조는 `Dataset → DataLoader → Model → Loss → Train loop → Sampling`입니다.

## 1. 데이터 준비

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

if not Path("names.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
chars = ['.'] + chars

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(stoi)
encoded_words = [[stoi[ch] for ch in w] for w in words]

print("num words:", len(words))
print("vocab_size:", vocab_size)
print(words[:10])

num words: 32033
vocab_size: 27
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']


In [ ]:
encoded_words

[[5, 13, 13, 1],
 [15, 12, 9, 22, 9, 1],
 [1, 22, 1],
 [9, 19, 1, 2, 5, 12, 12, 1],
 [19, 15, 16, 8, 9, 1],
 [3, 8, 1, 18, 12, 15, 20, 20, 5],
 [13, 9, 1],
 [1, 13, 5, 12, 9, 1],
 [8, 1, 18, 16, 5, 18],
 [5, 22, 5, 12, 25, 14],
 [1, 2, 9, 7, 1, 9, 12],
 [5, 13, 9, 12, 25],
 [5, 12, 9, 26, 1, 2, 5, 20, 8],
 [13, 9, 12, 1],
 [5, 12, 12, 1],
 [1, 22, 5, 18, 25],
 [19, 15, 6, 9, 1],
 [3, 1, 13, 9, 12, 1],
 [1, 18, 9, 1],
 [19, 3, 1, 18, 12, 5, 20, 20],
 [22, 9, 3, 20, 15, 18, 9, 1],
 [13, 1, 4, 9, 19, 15, 14],
 [12, 21, 14, 1],
 [7, 18, 1, 3, 5],
 [3, 8, 12, 15, 5],
 [16, 5, 14, 5, 12, 15, 16, 5],
 [12, 1, 25, 12, 1],
 [18, 9, 12, 5, 25],
 [26, 15, 5, 25],
 [14, 15, 18, 1],
 [12, 9, 12, 25],
 [5, 12, 5, 1, 14, 15, 18],
 [8, 1, 14, 14, 1, 8],
 [12, 9, 12, 12, 9, 1, 14],
 [1, 4, 4, 9, 19, 15, 14],
 [1, 21, 2, 18, 5, 25],
 [5, 12, 12, 9, 5],
 [19, 20, 5, 12, 12, 1],
 [14, 1, 20, 1, 12, 9, 5],
 [26, 15, 5],
 [12, 5, 1, 8],
 [8, 1, 26, 5, 12],
 [22, 9, 15, 12, 5, 20],
 [1, 21, 18, 15, 18, 1],
 

## 2. Dataset (`block_size = 1`)

In [ ]:
class NamesContextDataset(Dataset):
    def __init__(self, encoded_words, block_size):
        self.X, self.Y = [], []
        for word in encoded_words:
            context = [0] * block_size
            for ix in word + [0]:
                self.X.append(context.copy())
                self.Y.append(ix)
                context = context[1:] + [ix]
        self.X = torch.tensor(self.X, dtype=torch.long)
        self.Y = torch.tensor(self.Y, dtype=torch.long)

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

block_size = 1
dataset = NamesContextDataset(encoded_words, block_size)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

x, y = dataset[1]
xb, yb = next(iter(loader))
print("single x:", x, x.shape)
print("single y:", y)
print("batch xb.shape:", xb.shape)
print("batch yb.shape:", yb.shape)

single x: tensor([5]) torch.Size([1])
single y: tensor(13)
batch xb.shape: torch.Size([32, 1])
batch yb.shape: torch.Size([32])


## 3. Bigram model (명시적 one-hot 버전)

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.W = nn.Parameter(torch.randn(vocab_size, vocab_size))

    def forward(self, x):
        x = x.view(-1)  # (B,)
        x_onehot = F.one_hot(x, num_classes=self.vocab_size).float()  # (B, V)
        logits = x_onehot @ self.W                                    # (B, V)
        return logits

model = BigramLanguageModel(vocab_size)
logits = model(xb)
print("logits.shape:", logits.shape)
print("initial loss:", F.cross_entropy(logits, yb).item())

logits.shape: torch.Size([32, 27])
initial loss: 3.9422969818115234


In [ ]:
# @title
import torch
import torch.nn as nn
import torch.nn.functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, x):
        x = x.view(-1)  # (B,)
        logits = self.token_embedding_table(x)  # (B, V)
        return logits

## 4. Train / Eval 함수

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

## 5. 학습

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = BigramLanguageModel(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-1)

for epoch in range(20):
    train_loss = train_one_epoch(model, loader, optimizer, device)
    if epoch % 5 == 0 or epoch == 19:
        print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 2.5288
epoch  5 | train loss 2.5238
epoch 10 | train loss 2.5226
epoch 15 | train loss 2.5234
epoch 19 | train loss 2.5227


## 6. Sampling

In [ ]:
@torch.no_grad()
def sample(model, block_size, itos, device, num_samples=10, max_len=20):
    model.eval()
    results = []
    for _ in range(num_samples):
        context = torch.zeros((1, block_size), dtype=torch.long, device=device)
        out = []
        for _ in range(max_len):
            logits = model(context)
            probs = F.softmax(logits, dim=-1)
            ix = torch.multinomial(probs, num_samples=1)
            next_token = ix.item()
            if next_token == 0:
                break
            out.append(itos[next_token])
            context = torch.cat([context[:, 1:], ix], dim=1)
        results.append("".join(out))
    return results

sample(model, block_size, itos, device, num_samples=10)

['de',
 'aryavrite',
 'aeeghan',
 'aromorrabrqube',
 'mannn',
 'milamariel',
 'd',
 'yneraynamjond',
 'ashahariama',
 'viedra']

## 7. 정리

- `block_size=1`이면 bigram 문제를 정확히 표현할 수 있습니다.
- one-hot과 행렬곱만으로도 language model을 만들 수 있습니다.
- 이 학습 루프는 이후 단계에서도 거의 그대로 재사용됩니다.